# Preliminary 실험 종합 보고서

**실제 자료 탐색 · 의도적 오용 · 합성 효과 주입을 구분한다. 본 연구에 사용하지 않는다.**

각 노트북의 전체 결과와 가정을 읽어 비교한다. 여기서 새로운 회귀나 새로운 표본 탐색은 하지 않고, 실제자료 탐색의 전체 검정군 보정을 추가로 계산한다. 보고서는 canonical artifact로 저장한다.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json,hashlib,sqlite3,subprocess,shutil
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'metadata/experiments/lab_plan.json').exists())
OUT=ROOT/'results/experiments';REPORT_DIR=ROOT/'reports/experiments'
OUT.mkdir(parents=True,exist_ok=True);REPORT_DIR.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
load=lambda p:json.loads(p.read_text(encoding='utf-8'))
summaries={key:load(OUT/f'{key}_summary.json') for key in ['01','02','03','05','06','07']}
assert all(v['status']=='complete' for v in summaries.values())
paths={'grid':OUT/'01_specification_results.csv','ppml':OUT/'05_results.csv','seat':OUT/'07_results.csv',
 'misuse':OUT/'02_results.csv','power':OUT/'03_power.csv','core':OUT/'01_restored_core_areas.csv',
 'calibration':OUT/'03_null_calibration.csv'}
frames={k:pd.read_csv(p) for k,p in paths.items()}
input_hashes={p.relative_to(ROOT).as_posix():sha(p) for p in [*paths.values(),*[OUT/f'{k}_summary.json' for k in summaries]]}
connection=sqlite3.connect(':memory:');datasets={};sources=[];blocks=[];tables=[];charts=[]
GENERATOR='notebooks/experiments/analysis/04_실험결과_보고서.ipynb'


In [2]:
def dataset(key,frame,path,description):
    frame.to_sql(key,connection,index=False,if_exists='replace')
    query='SELECT * FROM "'+key+'"'
    result=pd.read_sql_query(query,connection)
    datasets[key]=json.loads(result.to_json(orient='records',force_ascii=False))
    sources.append({'id':key,'label':description,'path':path.relative_to(ROOT).as_posix(),
        'query':{'engine':'sqlite','language':'sql','sql':query,'description':description,
        'tables_used':[key],'files_used':[path.relative_to(ROOT).as_posix(),GENERATOR]}})
def md(key,body,source=None):
    item={'id':key,'type':'markdown','body':body}
    if source:item['sourceId']=source
    blocks.append(item)
def table(key,title,columns,sort):
    tables.append({'id':key,'title':title,'dataset':key,'sourceId':key,
        'defaultSort':{'field':sort,'direction':'asc'},
        'columns':[{'field':f,'label':label,'type':kind,**({'format':'number'} if kind=='number' else {})} for f,label,kind in columns]})
    blocks.append({'id':key+'_block','type':'table','tableId':key})
def bar(key,title,category,value,label,unit,subtitle):
    charts.append({'id':key+'_chart','title':title,'dataset':key,'sourceId':key,'type':'horizontalBar',
        'subtitle':subtitle,'showDescription':True,'valueFormat':'number','unit':unit,
        'encodings':{'x':{'field':category,'type':'nominal','label':'업종'},'y':{'field':value,'type':'quantitative','label':label}}})
    blocks.append({'id':key+'_chart_block','type':'chart','chartId':key+'_chart'})

## 1. 실제자료 탐색의 전체 검정군

520개 기준·단위 탐색, PPML20개, 좌석40개를 함께 기록한다. 의도적 오용의 무효 p값과 합성실험은 이 군에 넣지 않는다. 중복된10개 재적합도 시도기록에 남기므로 전체 시도와 고유 사양의 수는 다르다. BH 외에 의존구조와 무관한 Bonferroni 보정도 함께 표시한다. 어느 보정도 사후 선택된 연구 질문을 독립 검증으로 바꾸지는 않는다.

In [3]:
records=[]
for family,key in [('측정·표본·단위 탐색','grid'),('PPML','ppml'),('좌석 공급','seat')]:
    d=frames[key]
    for i,r in d.iterrows():
        spec=r.get('spec_id',r.get('scenario',''))
        q=r.get('p_bh_all520',r.get('p_bh20',r.get('p_bh40',np.nan)))
        records.append({'family':family,'row_index':i,'specification':spec,'industry':r['industry'],'outcome':r['outcome'],
            'exposure':r.get('exposure','scheduled_start'),'status':r['status'],'coefficient':r['coefficient'],
            'p_value':r['p_value'],'within_family_q':q})
real=pd.DataFrame(records)
real['p_bh_entire_search']=multipletests(real.p_value.fillna(1),method='fdr_bh')[1]
real['p_bonferroni_entire_search']=np.minimum(1,real.p_value.fillna(1)*len(real))
real_path=OUT/'04_all_real_exploratory_results.csv';real.to_csv(real_path,index=False,encoding='utf-8-sig')
family=real.groupby('family',sort=False).agg(attempts=('p_value','size'),estimated=('status',lambda s:int(s.eq('estimated').sum())),
 raw_significant=('p_value',lambda s:int(s.lt(.05).sum())),within_family_significant=('within_family_q',lambda s:int(s.lt(.05).sum())),
 entire_search_significant=('p_bh_entire_search',lambda s:int(s.lt(.05).sum())),min_raw_p=('p_value','min')).reset_index()
family_path=OUT/'04_family_summary.csv';family.to_csv(family_path,index=False,encoding='utf-8-sig')
dataset('families',family,family_path,'실제자료 탐색의 모든 시도. 가족별 보정과 실제자료 전체 탐색 보정을 구분. 의도적 오용·모의실험 제외.')
best=real.sort_values('p_value').groupby('family',sort=False).head(1).copy()
for k in ['p_value','within_family_q','p_bh_entire_search','p_bonferroni_entire_search']:best[k+'_text']=best[k].map(lambda x:f'{x:.6g}' if pd.notna(x) else '미추정')
dataset('best',best,real_path,'각 실험군에서 결과를 본 뒤 고른 최소p 사례. 확인적 결과나 독립 재현 아님.')
core=frames['core'].loc[frames['core'].spatial_definition.eq('centroid500')].copy()
core['label']=core.area_name+' / '+core.availability_threshold_pct.astype(str)+'%'
dataset('core',core,paths['core'],'중심점500m·공통매출·반복관측 이후 핵심상권 행수. 가용률 완화는 누락일정의 실제회차를 복원하지 않음.')
power_table=frames['power'].pivot(index='industry',columns='injected_pct',values='bh10_reject_pct').reset_index()
power_table.columns=['industry']+['effect_'+str(c) for c in power_table.columns[1:]]
dataset('power',power_table,paths['power'],'실제 관측 결과 아님. 지정한가상효과·설계·wild잔차분포에서500회 반복의업종별BH10 기각률.')
for key in summaries:
 sources.append({'id':'summary_'+key,'label':key+' 실험 명세','path':(OUT/f'{key}_summary.json').relative_to(ROOT).as_posix()})
print('실제자료 계수검정 시도',len(real))

실제자료 계수검정 시도 580


## 2. 의도적 오용과 다른 추정대상 구분

고정효과를 제거한 지역 간 관계는 원래의 동일지역 변화 질문과 다르다. 행 복제를 독립 표본으로 취급하거나 결과에 따라 관측을 삭제하는 것은 유효한 본 연구 개선이 아니다. 이 표들의 p값은 문제를 드러내기 위한 명목값이다.

In [4]:
bad=frames['misuse']
method_labels={'TWFE_cluster':'기준: 고정효과+군집','TWFE_iid':'고정효과+군집 무시',
 'quarterFE_cluster':'상권 고정효과 제거+군집','pooled_cluster':'상권·분기 고정효과 제거+군집'}
counts=bad.loc[bad.experiment.eq('A')].groupby('model').agg(attempts=('p_value','size'),nominal_significant=('nominal_p_lt_05','sum')).reset_index()
counts['label']=counts.model.map(method_labels)
dataset('bad_methods',counts,paths['misuse'],'같은기초표본20계수. 고정효과제거는다른추정대상, 군집무시는불확실성가정훼손. 명목유의수는발견수가아님.')
rep=bad.loc[bad.experiment.eq('B')&bad.industry.eq('커피-음료')&bad.outcome.eq('log1p_total_sales_won')].copy()
rep['method']=rep.model.map({'replicated_TWFE_iid':'중복을 독립 관측으로 취급','replicated_TWFE_original_cluster':'원래 상권 군집 유지'})
rep['label']=rep.replication_factor.astype(str)+'배 / '+rep['method']
rep['minus_log10_p']=-rep.log10_p
rep['beta_text']=rep.coefficient.map(lambda x:f'{x:.6f}')
dataset('replication',rep,paths['misuse'],'미리정한커피총매출의같은882관측복제시연. 명목행수만증가. iid p값은잘못된독립성가정으로만든수치.')
trim=bad.loc[bad.experiment.eq('C')].copy();trim['removed_pct']=100*trim.trim_fraction
trim['geometric_contrast_pct']=100*np.expm1(trim.coefficient*np.log(2))
trim['label']=trim.removed_pct.map(lambda x:f'{x:g}% 삭제')
dataset('trim',trim,paths['misuse'],'커피총매출에서결과기반으로음의잔차곱관측을우선삭제. 선택후통상p값은유효하지않음. 실제매출값을새로수정한것은아님.')
ppml=frames['ppml'].copy();ppml['outcome_label']=ppml.outcome.map({'total_sales':'총매출','night_sales':'21~24시 매출'})
ppml['effect_ci']=ppml.apply(lambda r:f"{r.doubling_pct:+.2f}% [{r.doubling_ci_low:+.2f}, {r.doubling_ci_high:+.2f}]" if r.status=='estimated' else '미추정',axis=1)
for k in ['p_value','p_bh20']:ppml[k+'_text']=ppml[k].map(lambda x:f'{x:.6g}')
dataset('ppml',ppml,paths['ppml'],'PPML20개전체. (1+예정회차)2배조건부산술평균상대차이,업종별상권군집t(G-1). 선택된조건부기하평균OLS와목표가다름.')


## 3. 원자료와 모의실험을 구분한 서술

원시 p값과 전체 탐색 보정값을 함께 제시한다. 가장 작은 p값은 탐색 후 선택된 예시이며, 새로운 확인적 가설이 아니다.

In [5]:
title='유의한 결과는 어떻게 만들어지는가: preliminary 실험'
md('title','# '+title+chr(10)+chr(10)+'**TEST 전용 · 본 연구에 사용하지 않음**  '+chr(10)+'종로구·마포구 · 외식업10종 · 2023~2025년 분기 · 2026년10월5일')
# 최종 결과와 좌석 감사가 완성된 뒤 summary를 title 바로 다음에 삽입한다.
md('scope',"""## 같은 자료로 다른 p값이 나오는 경로를 비교했다

목적은 유의한 행만 골라 본 연구로 옮기는 것이 아니라, **측정·표본·추정대상을 바꾼 결과와 통계적 가정을 훼손해 만든 결과를 구별**하는 것이다. 원자료와 기존 최종 분석은 그대로 보존했다. 분기 매출을 일별 소비로 만들거나 미제공 매출을0으로 채우지 않았다.

실제자료 실험은 상권 중심점/경계500m, 일정 가용률100·95·90·80%, 공연 ID/예정회차, 총매출/야간금액/야간비중과 화폐단위를 바꾼520개 계수검정이다. 여기에 비음수 금액용 PPML20개, 공식 개별공연장 좌석을 연결한40개를 추가했다. 의도적 오용286개와 합성효과 실험은 별도로 기록했다.

모든 결과는 이미 본 자료를 다시 탐색한 것이다. 전체 검정군 보정값도 보여주지만, 서로 겹치는 사양의 p값은 독립적이지 않다. 전체 BH는 의존조건을 검증하지 않은 탐색적 요약이며 FDR 보장을 주장하지 않는다. 임의 의존에서도 각 개별 p값이 유효하면 적용할 수 있는 Bonferroni도 함께 제시했다. 보정이 사후 선택된 연구 질문을 독립 검증으로 바꾸지는 않는다.""")
md('real_heading','## 실제자료에서 작은 p값은 나타났지만 탐색 범위를 함께 봐야 한다'+chr(10)+chr(10)+'각 가족 안의 보정과 전체 실제자료 탐색 보정을 구분했다. 가족별 유의 항목이 있더라도 다른 기준·모형도 함께 시도했다는 사실을 생략하면 안 된다. 520개에는 재현 확인용으로 중복한10개 결과도 포함되어 있다.')
table('families','실제자료 탐색 전수 요약',[
 ('family','실험군','text'),('attempts','시도한 계수','number'),('estimated','추정 완료','number'),('raw_significant','원 p<0.05','number'),
 ('within_family_significant','가족 BH<0.05','number'),('entire_search_significant','전체 BH<0.05','number')],'family')
md('minimums','아래는 **각 실험군에서 결과를 본 뒤 고른 최소 p 사례**다. 가장 좋은 결과만 보고했을 때 얼마나 강해 보이는지 보여주기 위한 표이며, 사전 지정된 주 결과가 아니다. 노출·결과변수·표본이 달라 계수 크기를 직접 순위화할 수 없다.')
table('best','탐색 후 선택한 최소 p 사례 — 확인적 발견 아님',[
 ('family','실험군','text'),('industry','업종','text'),('outcome','결과변수','text'),('p_value_text','원 p','text'),
 ('within_family_q_text','가족 BH','text'),('p_bh_entire_search_text','전체 BH','text'),('p_bonferroni_entire_search_text','전체 Bonferroni','text')],'family')
md('measurement_result',"""기준 탐색의 최소 p는 **경계 최단거리500m·가용80%·예정회차·치킨 야간비중**에서 나왔다. 1+예정회차2배에 대해 야간비중 +0.675%p, 개별95%구간[+0.257,+1.094], 원 p=.001765다. 해당10업종만 보면 BH=.01765지만, 같은공간·가용률·노출의 세결과30개를 함께 보면 .05296, 전체520개에서는 .62843이다.

이 관계는 야간 매출액 증가나 인과효과와 다르다. 총매출 변화로도 비중이 변할 수 있고,80% 기준은 미가용 공연의 실제회차를 채워준 것이 아니다. 따라서 본 분석의 주 결과를 이 조합으로 바꿀 근거로 쓰지 않는다.""")
md('ppml_heading',"""## PPML은 검토할 만한 모형이지만 다른 질문에 답한다

PPML은 비음수 매출의 **조건부 산술평균**을 로그 연결함수로 모형화한다. 매출액이 포아송 분포를 따른다고 가정하는 것이 아니며, 평균 명세와 군집 추론의 가정은 필요하다. 로그+1 회귀가 다루는 1+매출(원)의 조건부 기하평균과 목표가 다르고, 큰 매출 관측이 추정에 기여하는 방식도 달라진다. [저자의 PPML 설명](https://personal.lse.ac.uk/tenreyro/lgw.html)

같은 중심점500m·가용90%의 기초6,791행·127상권을 사용했다. 야간 매출이 모든 관측기간0인 상권×업종29개·259행은 PPML 고정효과의 경계해 처리로 제외했고 추가 분리 문제를 점검했다. 따라서 야간 적합 표본은 기초표본과 같지 않다.

한식 총매출은 **+1.266% [개별95%구간 +0.528,+2.010], p=.000909, PPML20개 BH=.01818**이다. 이 한식 결과는0전용 그룹 제거 없이 추정됐다. 그러나 전체 탐색군의 보정과 모형 목표 차이도 함께 읽어야 한다. 유의해서 PPML을 채택하는 것이 아니라, 실제 질문이 평균 매출 수준이고0원 관측을 유지해야 할 때 타당성을 검토할 모형이다.""")
table('ppml','PPML의 전체20개 결과',[
 ('industry','업종','text'),('outcome_label','결과','text'),('effect_ci','평균 상대차이 [개별95% CI]','text'),
 ('p_value_text','원 p','text'),('p_bh20_text','PPML20 BH','text'),('nobs','적합 행','number'),('clusters','상권','number')],'industry')
md('currency',"""0원을 포함한 log(1+매출)은 원·천원·백만원 중 무엇을 쓰는지에 따라0과양수의 상대 가중이 달라진다. 이는 데이터를 다르게 본 효과이며 단순 단위표시 변경이 아니다. [Chen·Roth의 연구](https://www.jonathandroth.com/assets/files/LogUniqueHOD0_Draft.pdf)

이번 단위 실험40개에서는 원 p<0.05인 결과가 없었다. 원/천원/백만원은 같은6,791행을 비교했고, 양수 매출만 로그변환한 경우는6,070행·125상권으로 표본이 바뀌었다. PPML은 만원/백만원 재적합에서 기울기 불변성을 확인했다. 단위 안정성은 유의성과 별도로 평가할 수 있는 개선 기준이다.""")
md('core_heading',"""## 가용률 기준을 낮추면 핵심상권이 돌아오지만 누락노출은 남는다

중심점100%에서 빠졌던 대학로(혜화역)는90%에서 제공120행 중40행, 홍대입구역(홍대)은120행 전부가 돌아왔다. 대학로를12분기 모두 포함하려면 이번 격자에서는80% 기준이 필요했다. 아래 그래프는 반복 관측 필터 후 남은 행수다.

가용률을 낮춰 유의한 값을 찾는 것보다 어떤 후보 ID 때문에 탈락했는지 확인해야 한다. 회차나 좌석을 모르면서 부분합을 실제 전체노출로 취급하지 않는다.""")
bar('core','핵심상권의 관측 복원','label','rows_after_repeat','유지된 상권×업종×분기 행수','행','각상권 원제공120행 · 중심점500m · 2분기 반복관측 이후')
charts[-1]['encodings']['x']['label']='상권 / 일정 가용 기준'


## 4. 공식 좌석 연결과 핵심상권 누락 원인

시설 전체 좌석과 개별 공연장 좌석을 구분한다. 현재 좌석을 과거 실제 관객으로 해석하지 않는다.

In [6]:
audit=summaries['06']
coverage=pd.DataFrame(audit['performance_ID_coverage'])
coverage=coverage.loc[coverage.scope.eq('study_candidate_performance_IDs')&coverage.metric.isin(['exact_hall_id_match','current_hall_seat_available'])].copy()
coverage['metric_label']=coverage.metric.map({'exact_hall_id_match':'공식 개별 공연장 ID 연결','current_hall_seat_available':'양의 정수 좌석 확인'})
dataset('seat_coverage',coverage,OUT/'06_summary.json','연구 후보12,839공연ID 기준. 공식같은시설/같은공연장ID와 현재양의좌석 연결 가용률.')
causes=pd.DataFrame(audit['core_exclusion_causes'])
reason_labels={'no_regular_start_in_source_quarter':'해당 분기 정규 시작시각 없음','overlap_with_valid_runtime':'공연시간을 적용하면 회차가 겹침',
 'schedule:hol_only_no_normal_schedule':'공휴일 일정만 기재','schedule:missing_schedule':'일정 없음',
 'schedule:overlapping_weekday_blocks':'요일 블록 중복','schedule:reverse_weekday_range':'역방향 요일 범위','schedule:unsupported_schedule':'지원하지 않는 일정 형식'}
causes['reason']=causes.primary_exclusion_reason.map(reason_labels).fillna(causes.primary_exclusion_reason)
dataset('core_causes',causes,OUT/'06_summary.json','기존 일정파서에서 탈락한 ID×분기의 주 사유. 코드규칙의 원인이지 원문의 사실오류 확정이 아님.')
seat=frames['seat'].copy()
for c in ['p_value','p_bh40']:seat[c+'_text']=seat[c].map(lambda v:f'{v:.6g}' if pd.notna(v) else '미추정')
seat['effect_ci']=seat.apply(lambda r:f"{r.doubling_pct:+.2f}% [{r.doubling_ci_low:+.2f}, {r.doubling_ci_high:+.2f}]" if r.status=='estimated' else '미추정',axis=1)
seat['exposure_label']=seat.exposure.map({'known_joint_scheduled_starts':'예정회차','known_current_hall_seat_start_product':'예정회차×현재 개별 좌석'})
seat['outcome_label']=seat.outcome.map({'log1p_total_sales_won':'총매출 log(1+원)','log1p_night_sales_21_24_won':'야간 매출 log(1+원)'})
dataset('seat',seat,paths['seat'],'동일공통표본에서 일정회차와 현재개별hall좌석 가중회차 비교. 40검정전수. 실제좌석공급/관객수 아님.')
md('seat_heading','## 개별 공연장 좌석은 연결할 수 있지만 실제 집객력은 아직 모른다'+chr(10)+chr(10)+
 '고정 보관된 XML에는 시설 안의 개별 공연장 ID와 좌석 수가 있었다. 연구 후보12,839개 공연은 같은 시설의 공식 개별 공연장 ID로 모두 연결됐고, 양의 정수 좌석 값은11,504개(89.6%)에서 확인됐다. 좌석0으로 기재된 공연장은 미가용으로 두었다. 시설 이름을 이용한 추정 연결이나 좌석0 대체는 하지 않았다.')
table('seat_coverage','공식 ID와 좌석 정보의 가용 범위',[('metric_label','조건','text'),('numerator','확인 ID','number'),('denominator','후보 ID','number'),('pct','가용률(%)','number')],'metric_label')
md('seat_risk',"""복합시설 관련4,726개 공연에서 시설 전체 좌석/개별 공연장 좌석 비율은 중앙값2.0배, 최대57.15배였다. 시설 전체 좌석을 매 공연에 곱하면 공급 규모를 과대 계산할 수 있다.

이번 좌석 노출은 **일정을 읽을 수 있고 양의 좌석이 연결되는 공연의 예정회차×현재 개별 공연장 좌석 합계**다. 실제 상연·점유율·관객을 확인한 값이 아니고, 2023~2025년 당시 좌석 수가 현재와 같았는지도 확인하지 못했다. 큰 원문 좌석 값에는 경기장66,704석이 포함되며, 결과를 보고 자르거나 임의 상한을 적용하지 않았다.

비교는 중심점500m, 공동 가용률90%, 같은 관측행에서 회차만 센 지표와 좌석을 곱한 지표를 각각 추정하는40개 검정으로 고정했다. 좌석 가용 조건을 추가하면 홍대 같은 핵심상권의 가용률이 다시 내려갈 수 있다. 따라서 일정90% 실험과 좌석90% 실험은 같은 표본이 아니다.""")
seat_best=seat.loc[seat.p_value.idxmin()]
md('seat_result',f"좌석 비교40개 중 원 p<0.05는 {int(seat.p_value.lt(.05).sum())}개, 이40개 내 BH<0.05는 {int(seat.p_bh40.lt(.05).sum())}개였다. 아래 전수표는 두 노출의 결과를 같은 표본에서 비교한다. 1+노출이2배일 때 1+매출(원)의 조건부 기하평균 대비값이며, 회차2배와 좌석 공급지표2배는 서로 다른 변화다. 공통 적합표본은4,714행·113상권이며, 대학로(혜화역)와 홍대입구역(홍대)은 모두 제외됐다. 좌석 가중 총매출에서 양식 +3.04%, 패스트푸드 +3.12%가 보정 전 유의했지만 두 결과의 BH40은 약0.481이었다. 좌석을 연결했다는 사실 자체가 실제 집객력 측정을 검증하지는 않는다.")
table('seat','같은 표본의 회차·좌석 노출 비교40개',[('industry','업종','text'),('exposure_label','노출','text'),('outcome_label','결과변수','text'),
 ('effect_ci','대비값 [개별95% CI]','text'),('p_value_text','원 p','text'),('p_bh40_text','40개 BH','text'),('nobs','행','number'),('clusters','상권','number')],'industry')
md('cause_heading',"""## 먼저 확인할 것은 핵심상권에서 빠진 일정의 원문이다

대학로(혜화역)는 후보4,788 ID×분기 중522개가 기존 일정 규칙에서 제외됐다. 이 중384개(73.6%)는 러닝타임을 적용했을 때 회차가 겹치는 경우이고,110개(21.1%)는 요일 블록 중복이다. 이는 원문 또는 파서가 모호함을 뜻하며, 실제 공연이 불가능하다고 확정한 것이 아니다.

홍대입구역(홍대)은2,883개 중54개가 제외됐고, 공휴일 일정만 있는36개가 가장 많았다. 공식 좌석까지 요구하면 공동 가용 ID×분기는2,040개로 줄었다. 일정만 복원해도 좌석 노출의 누락 문제는 따로 남는다.

아래 표의 사유를 기준으로188개 공연·상권 점검 목록을 만들었다. 제외를 해제하기 전에 원문의 요일별 회차와 러닝타임을 검증하는 것이 우선이다.""")
table('core_causes','핵심상권의 기존 일정 제외 사유',[('area_name','상권','text'),('reason','주 사유','text'),('excluded_ID_quarters','제외 ID×분기','number'),('share_of_excluded_ID_quarters_pct','해당 상권 제외 중(%)','number')],'area_name')


## 5. 의도적으로 작은 p값을 만드는 시연

지역 간 관계, iid 추론, 행 복제, 결과 기반 제외를 구분한다. 잘못된 p값은 본 연구의 발견으로 사용할 수 없다.

In [7]:
md('misuse_heading',"""## p값을 줄이는 것 자체는 가능했다

다음 실험은 **좋은 분석을 추천하기 위한 결과가 아니라, p값만을 목표로 삼을 때 생기는 문제를 드러내는 시연**이다. 실제자료 기준 탐색과 합쳐 유의한 결과 수를 세지 않았다.

같은20개 계수검정에서 상권·시기 고정효과와 상권 군집 추론을 유지하면 원 p<0.05는0개였다. 군집을 무시하면2개, 상권 고정효과를 제거하면9개였다. 고정효과 제거 결과는 지역 내부의 변화와 지역 간 차이를 섞은 다른 질문이므로, 그 자체를 계산 오류라고 부르지는 않는다. 다만 기존 질문의 근거로 대체할 수는 없다.""")
bar('bad_methods','추론·고정효과 변경의 명목 유의 개수','label','nominal_significant','명목 p<0.05 결과','개','각20개 계수 · 추정대상 변경과 잘못된 불확실성 계산을 구분')
md('replication_heading',"""### 같은 행을 복제하면 정보 없이 유의성만 커진다

공동표본5,015행을 동일 배수로 복제한다고 가정한 대수 계산에서, 커피 총매출882개 고유 관측의 결과를 살펴봤다. 대량 복제행을 실제 생성하지 않고 계수와 분산의 수식으로 계산했으며, 소규모 물리 복제로 검증했다. 계수는−0.015126으로 같지만, 독립 표본으로 잘못 세면 p가0.414에서 약1.97×10⁻¹⁷⁰으로 내려갔다. 원상권 ID로 군집화하면 p는 약0.665에서0.641로만 변했다. 작은 차이는 유한표본 보정값 변화이며 정보가 추가됐기 때문이 아니다.

아래 표에서 고유 관측과 명목 행 수를 구분해 읽어야 한다. 행 복제는 자료 확충이나 개선이 아니다.""")
table('replication','행 복제 시 계수와 p값',[('replication_factor','복제 배수','number'),('method','표준오차 계산','text'),('unique_rows_industry','고유 행','number'),('nominal_rows_industry','명목 행','number'),('beta_text','계수','text'),('p_display','계산된 p','text')],'replication_factor')
md('trim_heading',"""### 원하는 방향과 어긋나는 관측을 빼면 부호도 바뀐다

커피 총매출에서 기준모형의 양의 관계에 불리한 관측을 결과를 보고 순서화한 뒤 일부를 제외했다.882행 중17행(약1.93%, 목표2%)을 빼자 계수는−0.01513에서+0.04104로 바뀌고 naive p=.01013이 됐다.176행(약19.95%)을 빼면 p는 약1.47×10⁻¹⁸까지 작아졌다.

**이 p값과 구간은 결과를 보고 표본을 골랐다는 사실을 반영하지 않아 검증 추론으로 무효다.** 데이터 오류를 확인한 정당한 정제와 다르다. 표는 제외한 비율별 결과를 전부 보여준다.""")
table('trim','결과를 본 뒤 관측 제외 — p값은 검증용으로 무효',[('removed_pct','목표 제외(%)','number'),('rows_removed_by_selection','제외한 행','number'),('unique_rows_industry','남은 고유 행','number'),('coefficient','계수','number'),('p_display','naive p (무효)','text')],'removed_pct')
md('p_guidance','작은 p값은 효과 크기나 측정의 타당성을 보장하지 않는다. 분석 선택과 시도한 결과를 함께 공개해야 한다. [미국통계학회 p값 성명](https://www.amstat.org/asa/files/pdfs/p-valuestatement.pdf)')


## 6. 합성 효과와 검출 가능성

실제 회귀관계를 제거한 잔차에 정해진 효과를 주입한다. 실제 공연 효과를 발견한 결과가 아니다.

In [8]:
cal=frames['calibration'].iloc[0]
md('simulation_heading',"""## 효과가 존재한다고 가정해도 업종별 검출 가능성은 다르다

현재 기준표본5,015행·114상권의 구조와 잔차를 이용해 기존 회귀관계를 제거하고, **1+예정회차2배에 대해 1+총매출(원)의 조건부 기하평균 대비가0·2·5·10·20%가 되도록 합성 효과를 주입**했다. 이는 실제 매출에 그 효과가 있다는 증거가 아니다.

상권별로 같은 무작위 부호를10업종에 공유하고 고정효과를 다시 제거하는 실험을500회 반복했다. 아래 값은10업종 BH 보정 후 해당 업종이 기각된 비율이다. 모든 업종에 동일한 효과를 주입했고, 실제 자료에 없는 신규 정보를 추가하지 않았다.

5% 합성 효과에서도 업종별 검출 비율은18.2~100%로 갈렸다.10%에서는55.2~100%,20%에서는 모든 업종이100%였다. 현재 구조가 어떤 크기의 신호를 잘 놓칠 수 있는지 보여주는 조건부 시연이다. 표의0% 열은 업종별 오탐 빈도이며 가족 전체 FDR 자체가 아니다.""")
table('power','합성 효과별 BH10 기각률 — 관측된 공연 효과 아님',[('industry','업종','text')]+[(c,c.replace('effect_','')+'% 주입','number') for c in power_table.columns if c!='industry'],'industry')
md('null_calibration',f"전체 효과를0으로 둔500회에서 BH10 후 하나라도 기각한 비율은 {cal.any_BH10_rejection_pct:.1f}%였다. 원 p값 기준 업종별 기각률은 {cal.industry_raw_rejection_min_pct:.1f}~{cal.industry_raw_rejection_max_pct:.1f}%여서 명목5% 검정 크기가 업종마다 보장되지는 않았다. 검출 비율의 몬테카를로 구간도 별도 전수 결과에 저장했다. 현재 추정잔차, 고정된 설명변수, 상권 간 독립 부호 가정에 조건부이며 공간 상관과 노출 측정오차를 새로 모형화하지 않았다. 따라서 미래 연구의 확정 검정력이나 필요 표본 수로 읽을 수 없다.")
md('method_heading',"""## 해석 범위와 본 분석에서 검토할 개선

실제자료 격자와 좌석 비교는 상권×업종 및 업종×연도분기 고정효과를 사용하고 상권별 군집 표준오차를 적용했다. PPML은 업종별 상권·분기 고정효과와 상권 군집 추론으로 조건부 산술평균을 추정했다. 최소2분기 반복 관측과 최소20상권 등의 추정 조건을 점검했다. 서로 다른 모형의 1+매출(원)의 기하평균 대비와 매출 산술평균 대비는 같은 효과라고 비교하지 않는다.

- **일정 제외 원문부터 검증한다.** 대학로의 회차·러닝타임 겹침과 요일 중복을 점검하면 핵심상권 탈락의 원인을 직접 다룰 수 있다. 유의성을 기준으로 제외 규칙을 느슨하게 만들지는 않는다.
- **공연 규모를 재려면 공식 개별 공연장 좌석을 연결한다.** 예정회차×현재 좌석은 현실적인 중간 지표지만 실제 관객과 구분하고 과거 좌석 정보·공연별 판매 가능 좌석을 추가 확인한다.
- **질문이 평균 매출 수준이면 PPML을 검토한다.**0원 관측과 화폐단위 안정성이라는 이유로 선택할 수 있다. 이번 한식 p값을 보고 선택했다는 사실을 숨겨 확인적 주 결과로 만들지는 않는다.
- **새 자료에서 판단 기준을 고정한다.** 이미 본 자료에서 업종·기간·노출·모형을 고른 뒤에는 같은 자료의 보정 p값만으로 독립 검증을 대신할 수 없다. 향후 기간을 확보할 경우 분석 기준을 먼저 고정한다.

분기 매출만으로 공연 직후3시간·6시간의 실제 소비를 식별할 수 없다는 한계는 남는다. 실제 관객·체류·소비로 이어지는 중간 경로와 인과관계도 이번 실험에서 확인하지 않았다. 좌석을 정확히 연결했는지와 그 좌석이 소비 수요를 제대로 나타내는지는 별도의 문제다.""")
md('remaining','## 남은 확인 사항'+chr(10)+chr(10)+'대학로의 회차 겹침은 일정 해석 문제인지, 개별 공연장의 과거 좌석이 현재와 같은지, PPML의 평균 명세와 상권 간 공간 상관이 결과에 어떤 영향을 주는지가 남아 있다. 본 연구에서는 이 측정·모형 문제를 먼저 정리하고, 결론이 유의하든 아니든 같은 기준으로 보고해야 한다.')
summary_body=f"""## 실험에서 얻은 답

**작은 p값은 만들 수 있었지만, 본 분석의 개선 후보는 작은 p값 자체와 구분된다.** 실제자료 {len(real)}개 계수검정 중 원 p<0.05는 {int(real.p_value.lt(.05).sum())}개, 전체 BH 보정 후에는 {int(real.p_bh_entire_search.lt(.05).sum())}개, 전체 Bonferroni 보정 후에는 {int(real.p_bonferroni_entire_search.lt(.05).sum())}개였다. PPML 한식 총매출은20개 내 보정에서는 유의했으나, 더 넓은 탐색 범위와 다른 추정대상을 함께 봐야 한다.

행 복제나 결과를 보고17개 관측을 제외하는 것만으로도 매우 작은 p값을 만들 수 있었다. 이 결과는 개선의 증거가 아니다. 반면 **개별 공연장 좌석 연결, 핵심상권의 일정 제외 원인 검증, 평균 매출에 맞는 PPML 검토**는 자료와 연구 질문에 근거해 발전시킬 수 있는 후보다."""
blocks.insert(1,{'id':'summary','type':'markdown','body':summary_body})


## 7. 보고서 저장과 검증

모든 표·차트의 수치와 SQL 조회 근거를 canonical artifact에 보존하고 단일 HTML 보고서로 만든다. 본 연구 파일의 SHA-256을 다시 확인한다.

In [9]:
generated=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
artifact={'surface':'report','manifest':{'version':1,'surface':'report','title':title,'generatedAt':generated,
 'description':'실제자료 탐색·의도적 유의성 부풀리기·합성 효과 주입을 구분한 preliminary 실험 보고서',
 'blocks':blocks,'tables':tables,'charts':charts,'sources':sources},
 'snapshot':{'version':1,'generatedAt':generated,'status':'ready','datasets':datasets,'accessIssues':[]},'sources':sources}
assert all(sha(ROOT/path)==value for path,value in input_hashes.items())
assert len(real)==580 and len(frames['misuse'])==286
assert len({b['id'] for b in blocks})==len(blocks)
assert all(t['dataset'] in datasets for t in tables)
(OUT/'artifact.json').write_text(json.dumps(artifact,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
notes={'audience':'technical','delivery':'portable_html','question':'유의성 탐색에서 측정·표본·추정법의 개선 후보와 무효한 p값 생성 경로를 구분할 수 있는가?',
 'exploratory_after_prior_results':True,'real_attempts':len(real),'source_sha256':input_hashes,'baselines_modified':False,
 'required_structure_mapping':{'title':'title','technical_summary':'summary','definitions':'scope/ppml_heading/seat_risk/method_heading',
 'findings':'real_heading/ppml_heading/core_heading/seat_heading/cause_heading/misuse_heading/simulation_heading',
 'methodology':'scope/method_heading/simulation_heading','uncertainty':'minimums/currency/seat_risk/null_calibration','next_steps':'method_heading','further_questions':'remaining'},
 'chart_map':[{'id':'core_chart','family':'horizontalBar','fields':['label','rows_after_repeat'],'claim':'가용률 기준별 핵심상권 복원'},
 {'id':'bad_methods_chart','family':'horizontalBar','fields':['label','nominal_significant'],'claim':'다른 추정대상과 잘못된 불확실성 계산에서 명목 유의 개수가 바뀜'}],
 'table_rationale':'계수·CI·보정 p값, 잘못된 추론의 조건과 전체40·20검정은 정확한 조회가 중요하여 표로 제공. 합성효과는 업종과주입효과의 교차표로 표시. 두 막대차트는각각 표본지원과 방법별개수 비교라는 별개 질문.',
 'sharing_assessment':'PRELIMINARY_ONLY; no confirmatory discovery; invalid and synthetic outputs explicitly labelled',
 'multiple_testing':'BH entire580 is diagnostic under dependence; Bonferroni additionally shown. Ten duplicated checks retained as attempts. No correction makes data-guided selection confirmatory.',
 'hypothesis_search_scope':'520 specification +20 PPML +40 seat comparisons; misuse286 and simulations excluded',
 'validation':{'real_attempts':len(real),'raw_below05':int(real.p_value.lt(.05).sum()),'bh_entire_below05':int(real.p_bh_entire_search.lt(.05).sum()),'bonferroni_entire_below05':int(real.p_bonferroni_entire_search.lt(.05).sum())}}
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
print('artifact 저장:',len(tables),'표 /',len(charts),'차트')


artifact 저장: 9 표 / 2 차트


In [10]:
builder=Path('C:/Users/mamap/.codex/plugins/cache/openai-curated-remote/data-analytics/0.2.10-13ceeea1f599/skills/build-report/scripts/deliver_portable_artifact.mjs')
result_build=subprocess.run([shutil.which('node'),str(builder),'--input',str(OUT/'artifact.json'),'--output',str(REPORT_DIR/'report.html')],capture_output=True,text=True,encoding='utf-8',timeout=120)
print(result_build.stdout[-6000:])
if result_build.returncode:print(result_build.stderr[-6000:])
assert result_build.returncode==0
(OUT/'delivery_receipt.json').write_text(result_build.stdout,encoding='utf-8')
receipt=json.loads(result_build.stdout.strip());notes['delivery_verification']=receipt['stages']['verification']
notes['browser_visual_qa_performed']=notes['delivery_verification']=='passed'
notes['qa_limitation']='자동 브라우저 화면검사 불가. 수치·문서구조·의미적 대체표 검증 완료.' if not notes['browser_visual_qa_performed'] else ''
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
plan=load(ROOT/'metadata/experiments/lab_plan.json')
assert all(sha(ROOT/path)==value for path,value in input_hashes.items())
assert all(sha(ROOT/path)==value for path,value in plan['protected_sha256'].items())
print('보고서 완료 / 본 연구 파일 불변',len(plan['protected_sha256']))


{"ok":true,"html":"C:\\Coding\\졸업프로젝트\\reports\\experiments\\report.html","stages":{"validation":"passed","package":"passed","verification":"structural_only"},"browserWarning":{"code":"browser_unavailable","message":"No installed Chromium headless-shell executable was found. Set CHROMIUM_EXECUTABLE_PATH (or the legacy PLAYWRIGHT_EXECUTABLE_PATH) or preinstall Chromium headless-shell under PLAYWRIGHT_BROWSERS_PATH; portable tooling never downloads browsers."},"counts":{"blocks":32,"charts":2,"html":0,"metrics":0,"tables":9},"sourceDialog":"not_verified","sourceInteraction":"not_verified","viewports":[],"timings":{"validateAndPackageMs":159.2,"staticChartExtractionMs":66.2,"structuralVerificationMs":9.3,"totalMs":177.5}}

보고서 완료 / 본 연구 파일 불변 305
